In [2]:
import pandas as pd
import seaborn as sns
from utils import parse_har_files_to_dataframe
from common import BASE_HAR_DIR


from LeakDetector import LIKELY_HASHES, ENCODINGS_NO_ROT, LeakDetector

sns.set_theme(style="whitegrid", context="paper")
pd.set_option('display.max_colwidth', 128)

df = parse_har_files_to_dataframe(BASE_HAR_DIR, test=False, use_cache=True)


In [3]:
import warnings

from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=(SettingWithCopyWarning))


### Load the SST hostnames to use in leak detection

In [5]:
df_sst_hosts = pd.read_csv('detected_sst_req_hostnames.csv')
sst_req_hostnames = set(df_sst_hosts.req_hostname.unique())

In [6]:
def search_prod_name(search_term, site_domain, exp_mode, consent_mode, df, sst_req_hostnames):
    # Initialize a leak detector with the product name as the search term
    print(f"Searching: {search_term} in {site_domain} | {exp_mode} - {consent_mode}")

    # Initialize LeakDetector instance
    ld = LeakDetector(
        [search_term],
        encoding_set=ENCODINGS_NO_ROT,
        hash_set=LIKELY_HASHES,
        encoding_layers=2,
        hash_layers=0,
        debugging=False
    )

    # Filter DataFrame for the specific site_domain and conditions
    df_filtered = df[(df.site_domain == site_domain) &
                     (df.exp_mode == exp_mode) &
                     (df.consent_mode == consent_mode)]
    assert len(df_filtered) > 0, f"No data found for {site_domain} {exp_mode} - {consent_mode}"

    # Detect leaks in URLs and POST data
    df_filtered['url_leaks'] = df_filtered['url'].apply(ld.check_url, encoding_layers=2)
    df_filtered['post_leaks'] = df_filtered['post_data'].apply(ld.check_post_data, encoding_layers=2)

    # Helper function to create a leak record
    def create_leak_record(row, leak, leak_type, is_sst):
        return {
            'site_domain': row['site_domain'],
            'search_term': search_term,
            'leak_type': leak_type,
            'leak_encoding': leak[0] if len(leak) > 1 else '',
            'leak_value': leak[1] if len(leak) > 1 else leak[0],
            'consent_mode': consent_mode,
            'exp_mode': exp_mode,
            'country': row['country'],
            'req_hostname': row['req_hostname'],
            'is_tracker': row['is_tracker'],
            'is_cname_tracker': row['is_cname_tracker'],
            'sst': is_sst,
            'entity': row['entity'],
            'url': row['url'],
            'post_data': row['post_data'],
        }

    # Collect individual leaks
    leak_records = []
    for idx, row in df_filtered.iterrows():
        is_sst = row['req_hostname'] in sst_req_hostnames

        # Collect URL leaks
        leak_records.extend(create_leak_record(row, url_leak, 'url_leak', is_sst) for url_leak in row['url_leaks'])

        # Collect POST data leaks
        leak_records.extend(create_leak_record(row, post_leak, 'post_leak', is_sst) for post_leak in row['post_leaks'])

    return pd.DataFrame(leak_records)


In [7]:
def search_prod_name(seach_term, site_domain, exp_mode, consent_mode, df, sst_req_hostnames):
    # Initialize a leak detector with the product name as the search term
    print(f"Searching: {seach_term} in {site_domain} | {exp_mode} - {consent_mode}")

    # Initialize leak detector
    ld = LeakDetector(
        [seach_term],
        encoding_set=ENCODINGS_NO_ROT,
        hash_set=None,
        encoding_layers=2,
        hash_layers=0,
        debugging=False
    )

    # Filter DataFrame for the specific site_domain
    df_filtered = df[(df.site_domain == site_domain) &
                     (df.exp_mode == exp_mode) &
                     (df.consent_mode == consent_mode)]
    assert len(df_filtered) > 0, f"No data found for {site_domain} {exp_mode} - {consent_mode}"
    # Apply leak detection to URLs and post_data
    df_filtered['url_leaks'] = df_filtered['url'].apply(lambda url: tuple(ld.check_url(url, encoding_layers=2)))
    df_filtered['post_leaks'] = df_filtered['post_data'].apply(lambda post_data: tuple(ld.check_post_data(post_data, encoding_layers=2)))

    # Initialize list to store individual leak details
    leak_records = []

    # Iterate through the filtered DataFrame and collect individual leaks
    for idx, row in df_filtered.iterrows():
        is_sst = row['req_hostname'] in sst_req_hostnames
        # Collect each URL leak
        for url_leak in row['url_leaks']:
            leak_records.append({
                'site_domain': row['site_domain'],
                'search_term': seach_term,
                'leak_type': 'url_leak',
                'leak_encoding': url_leak[0] if len(url_leak) > 1 else '',
                'leak_value': url_leak[1] if len(url_leak) > 1 else url_leak[0],
                # add all the other columns
                'consent_mode': consent_mode,
                'exp_mode': exp_mode,
                'country': row['country'],
                # df_filtered.columns
                'req_hostname': row['req_hostname'],
                'is_tracker': row['is_tracker'],
                'is_cname_tracker': row['is_cname_tracker'],
                'sst': is_sst,
                'entity': row['entity'],
                'url': row['url'],
                'post_data': row['post_data'],
            })

        # Collect each POST data leak
        for post_leak in row['post_leaks']:
            leak_records.append({
                'site_domain':  row['site_domain'],
                'search_term': seach_term,
                'leak_type': 'post_leak',
                'leak_encoding': post_leak[0] if len(post_leak) > 1 else '',
                'leak_value': post_leak[1] if len(post_leak) > 1 else post_leak[0],
                'consent_mode': consent_mode,
                'exp_mode': exp_mode,
                'country': row['country'],
                # df_filtered.columns
                'req_hostname': row['req_hostname'],
                'is_tracker': row['is_tracker'],
                'is_cname_tracker': row['is_cname_tracker'],
                'sst': is_sst,
                'entity': row['entity'],
                'url': row['url'],
                'post_data': row['post_data'],
            })

    return pd.DataFrame(leak_records)



In [8]:

from tld import get_fld
from utils import CONSENT_MODES, MEASUREMENT_MODES

PRODUCT_NAMES_CSV = "product_names.csv"

dfl = pd.DataFrame(columns=[
    'site_domain', 'search_term', 'leak_type', 'leak_encoding',
    'leak_value', 'consent_mode', 'exp_mode', 'country', 'req_hostname',
    'is_tracker', 'is_cname_tracker', 'sst', 'entity', 'url', 'post_data'])

# read the product names from the CSV
prod_names = pd.read_csv(PRODUCT_NAMES_CSV)

# we only check the "tracking" hars
cnt = 0
# for exp_mode in MEASUREMENT_MODES:
for __, row in prod_names.iterrows():
    site_domain = get_fld(row['site'], fix_protocol=True)
    leak_df = search_prod_name(row['product_name'], site_domain, row['exp_mode'], row['consent_mode'], df, sst_req_hostnames)
    dfl = pd.concat([dfl, leak_df], ignore_index=True)
    cnt += 1
    if cnt > 100:
        print(f"Processed {cnt} product names")
        # break

dfl['req_domain'] = dfl['req_hostname'].apply(lambda x: get_fld(x, fix_protocol=True))
dfl.head()


Searching: CLEARBLUE Schwangerschaftstest frühe Erkennung in apotal.de | tracking - Accept
Searching: RedCare Schwangerschaftstest - Frühtest 1 St  in shop-apotheke.com | tracking - Accept
Searching: Clearblue Schwangerschaftstest Frühe Erkennung in sanicare.de | tracking - Accept
Searching: BE Sure Schwangerschaftstest 1 St  in aponeo.de | tracking - Accept
Searching: BE SURE Schwangerschaftstest in medikamente-per-klick.de | tracking - Accept
Searching: CLEARBLUE Schwangerschaftstest schnelle Erkennung 5 Stück in medpex.de | tracking - Accept
Searching: Clearblue Schwangerschaftstest Ultra Frühtest Dig in onfy.de | tracking - Accept
Searching: Be Sure Schwangerschaftstest 1 St in mycare.de | tracking - Accept
Searching: FRÜHTEST Schwangerschaftstest 1 St in docmorris.de | tracking - Accept
Searching: Schwangerschaftstest Schnelltest von apodiscounter 1 stk in apodiscounter.de | tracking - Accept
Searching: Zwangerschapstest Testjezelf.nu 3 stuks in gezondheidaanhuis.nl | tracking - A

,site_domain,search_term,leak_type,leak_encoding,leak_value,consent_mode,exp_mode,country,req_hostname,is_tracker,is_cname_tracker,sst,entity,url,post_data,req_domain
0,apotal.de,CLEARBLUE Schwangerschaftstest frühe Erkennung,url_leak,urlencode,CLEARBLUE Schwangerschaftstest frühe Erkennung,Accept,tracking,DE,bat.bing.com,True,False,False,Microsoft,https://bat.bing.com/action/0?ti=4024282&Ver=2&mid=803c3f28-c58f-4575-bdb8-4c588d08987c&sid=50e7c58088cc11ef8a4b2326ae1182b6...,,bing.com
1,apotal.de,CLEARBLUE Schwangerschaftstest frühe Erkennung,url_leak,urlencode,CLEARBLUE Schwangerschaftstest frühe Erkennung,Accept,tracking,DE,www.etracker.de,True,False,False,etracker,https://www.etracker.de/cntcc?&&et=3X9dHm&v=5.0&tc=17287597474736&pagename=CLEARBLUE%20Schwangerschaftstest%20fr%C3%BChe%20E...,,etracker.de
2,apotal.de,CLEARBLUE Schwangerschaftstest frühe Erkennung,url_leak,urlencode,CLEARBLUE Schwangerschaftstest frühe Erkennung,Accept,tracking,DE,www.etracker.de,True,False,False,etracker,https://www.etracker.de/cntcc?&&et=3X9dHm&v=5.0&tc=17287597474756&pagename=CLEARBLUE%20Schwangerschaftstest%20fr%C3%BChe%20E...,,etracker.de
3,apotal.de,CLEARBLUE Schwangerschaftstest frühe Erkennung,url_leak,urlencode,CLEARBLUE Schwangerschaftstest frühe Erkennung,Accept,tracking,DE,www.etracker.de,True,False,False,etracker,https://www.etracker.de/cntcc?&&et=3X9dHm&v=5.0&tc=17287597592596&pagename=CLEARBLUE%20Schwangerschaftstest%20fr%C3%BChe%20E...,,etracker.de
4,apotal.de,CLEARBLUE Schwangerschaftstest frühe Erkennung,post_leak,,CLEARBLUE Schwangerschaftstest frühe Erkennung,Accept,tracking,DE,www.etracker.de,True,False,False,etracker,https://www.etracker.de/api/v6/tracking/webEvents,"------WebKitFormBoundaryu94X1ZZVobs5oVfA\r\nContent-Disposition: form-data; name=""et""\r\n\r\n3X9dHm\r\n------WebKitFormBound...",etracker.de


In [9]:
dfl['third_party'] = dfl['req_domain'] != dfl['site_domain']
dfl.third_party.value_counts().to_frame()

,count
third_party,
True,874
False,219


In [10]:
dfl.sst.value_counts().to_frame()

,count
sst,
False,925
True,168


### Only take 3rd P or SST requests

In [12]:
dfl = dfl[dfl.third_party | dfl.sst]
dfla = dfl[dfl.consent_mode=='Accept']
dflr = dfl[dfl.consent_mode=='Reject']

## Top domains receiving product leaks

In [13]:

tmp_df = dfl.pivot_table(
    index='req_domain',                  # Rows: Unique request domains
    columns='consent_mode',           # Columns: Consent mode (accepted/rejected)
    values='site_domain',                # Values: Count of unique site domains
    aggfunc='nunique',                   # Aggregate function: Count unique site domains
    fill_value=0                          # Fill missing values with 0
)

tmp_df.sort_values(by='Accept', ascending=False).head(10)

consent_mode,Accept,Reject
req_domain,,
google.com,32,11
doubleclick.net,24,3
bing.com,21,4
google-analytics.com,12,17
googleadservices.com,7,1
tiktok.com,7,0
google.it,5,0
google.de,5,0
google.nl,5,1


In [14]:
print(tmp_df.sort_values(by='Accept', ascending=False).head(10).to_latex())

\begin{tabular}{lrr}
\toprule
consent_mode & Accept & Reject \\
req_domain &  &  \\
\midrule
google.com & 32 & 11 \\
doubleclick.net & 24 & 3 \\
bing.com & 21 & 4 \\
google-analytics.com & 12 & 17 \\
googleadservices.com & 7 & 1 \\
tiktok.com & 7 & 0 \\
google.it & 5 & 0 \\
google.de & 5 & 0 \\
google.nl & 5 & 1 \\
connectif.cloud & 5 & 4 \\
\bottomrule
\end{tabular}



## Top entities receiving product leaks (Table 5)

In [15]:

tmp_df = dfl[dfl.entity!=''].pivot_table(
    index='entity',                  # Rows: Unique request domains
    columns='consent_mode',           # Columns: Consent mode (accepted/rejected)
    values='site_domain',                # Values: Count of unique site domains
    aggfunc='nunique',                   # Aggregate function: Count unique site domains
    fill_value=0                          # Fill missing values with 0
)

tmp_df.sort_values(by='Accept', ascending=False).head(5)

consent_mode,Accept,Reject
entity,,
Google,36,23
Microsoft,23,7
ByteDance,7,0
Facebook,4,1
Virtual Minds,4,0


In [16]:
print(tmp_df.sort_values(by='Accept', ascending=False).head(5).to_latex())

\begin{tabular}{lrr}
\toprule
consent_mode & Accept & Reject \\
entity &  &  \\
\midrule
Google & 36 & 23 \\
Microsoft & 23 & 7 \\
ByteDance & 7 & 0 \\
Facebook & 4 & 1 \\
Virtual Minds & 4 & 0 \\
\bottomrule
\end{tabular}



## Leak encodings and location (URL or POST body)

In [17]:
dfl.groupby('leak_encoding')['site_domain'].nunique().sort_values(ascending=False).to_frame().head(10)

,site_domain
leak_encoding,
urlencode,47
,22
base64,2


In [19]:
dfl.leak_type.value_counts(normalize=True).to_frame()

,proportion
leak_type,
url_leak,0.774472
post_leak,0.225528


## Base64 encoded product name leaks

In [37]:
dfl[dfl.leak_encoding=='base64']

,site_domain,search_term,leak_type,leak_encoding,leak_value,consent_mode,exp_mode,country,req_hostname,is_tracker,is_cname_tracker,sst,entity,url,post_data,req_domain,third_party
5,shop-apotheke.com,RedCare Schwangerschaftstest - Frühtest 1 St,url_leak,base64,RedCare Schwangerschaftstest - Frühtest 1 St,Accept,tracking,DE,o.adtriba.com,True,False,False,,https://o.adtriba.com/collect?id=f38892cb-4d5c-4de9-b4a6-c05871a13e09&ts=1728756131079&data=eyJpZCI6IjNiYzQxOTkzLWJlMmQtNGE1...,,adtriba.com,True
693,redcare.it,Clearblue Test di Gravidanza con Indicatore delle Settimane 2 pz,url_leak,base64,Clearblue Test di Gravidanza con Indicatore delle Settimane 2 pz,Accept,tracking,IT,o.adtriba.com,True,False,False,,https://o.adtriba.com/collect?id=b6dbd66d-d805-4833-bf78-7c01e03e5a42&ts=1728839429910&data=eyJpZCI6IjVkY2ZhMDUwLWNmNDctNDBl...,,adtriba.com,True
783,shop-apotheke.com,Clearblue Schwangerschaftstest Ultra Frühtest Digital 2er,url_leak,base64,Clearblue Schwangerschaftstest Ultra Frühtest Digital 2er,Reject,tracking,DE,o.adtriba.com,True,False,False,,https://o.adtriba.com/collect?id=60005321-114c-48f6-ad22-b281b1a547af&ts=1728772753038&data=eyJpZCI6IjMxZjQxNjIzLTgwMjMtNDA0...,,adtriba.com,True
1056,redcare.it,Clearblue Test di Gravidanza con Indicatore delle Settimane 2 pz,url_leak,base64,Clearblue Test di Gravidanza con Indicatore delle Settimane 2 pz,Reject,tracking,IT,o.adtriba.com,True,False,False,,https://o.adtriba.com/collect?id=3de29d2f-80d9-4d85-bd08-ba9461989621&ts=1729083418693&data=eyJpZCI6IjlkZmRlMjRiLWI0N2ItNDc5...,,adtriba.com,True


In [22]:

tmp_df = dfl.pivot_table(
    index='country',                  # Rows: Unique request domains
    columns='consent_mode',           # Columns: Consent mode (accepted/rejected)
    values='site_domain',                # Values: Count of unique site domains
    aggfunc='nunique',                   # Aggregate function: Count unique site domains
    fill_value=0                          # Fill missing values with 0
)

tmp_df.sort_values(by='Accept', ascending=False).head(10)

consent_mode,Accept,Reject
country,,
DE,10,3
ES,10,9
IT,9,10
NL,8,9
FR,7,7


In [23]:
print(tmp_df.sort_values(by='Accept', ascending=False).to_latex())

\begin{tabular}{lrr}
\toprule
consent_mode & Accept & Reject \\
country &  &  \\
\midrule
DE & 10 & 3 \\
ES & 10 & 9 \\
IT & 9 & 10 \\
NL & 8 & 9 \\
FR & 7 & 7 \\
\bottomrule
\end{tabular}



In [24]:
dfl.groupby(['country', 'site_domain', 'leak_value', 'consent_mode'])['req_domain'].nunique().sort_values(ascending=False).to_frame().head(10)

req_domain
country site_domain        leak_value                                                       consent_mode            
ES      farmaciasdirect.es Alvita Test De Embarazo                                          Accept                12
IT      efarma.com         Clearblue Test di Gravidanza Rilevazione Rapida 2 Test           Accept                 9
        1000farmacie.it    Test Gravidanza Meds 1 Pezzo                                     Accept                 9
        redcare.it         Clearblue Test di Gravidanza con Indicatore delle Settimane 2 pz Accept                 8
NL      da.nl              DA Zwangerschapstest 2 stuks                                     Accept                 8
IT      farmasave.it       Clearblue Plus - Test di Gravidanza - 1 Stick                    Accept                 8
DE      apodiscounter.de   Schwangerschaftstest Schnelltest von apodiscounter 1 stk         Accept                 8
        docmorris.de       FRÜHTEST Schwangerschaftstest 1 St                               Accept                 8
IT      docpeter.it        Clearblue test gravidanza pregnancy visual 2 stick               Accept                 7
FR      docmorris.fr       Gestarelle Grossesse 30 Capsules                                 Accept                 7

## Reject leaks

In [26]:
dflr.groupby(['country', 'site_domain', 'leak_value'])['req_domain'].agg(lambda x: set(x)).sort_values(ascending=False).to_frame().head(10)

req_domain
country site_domain         leak_value                                                                                                                                                           
DE      apotal.de           CLEARBLUE Schwangerschaftstest frühe Erkennung                                                                                                {bing.com, etracker.de}
FR      universpharmacie.fr Clearblue Test de Grossesse Détection Rapide X2                                                                            {bing.com, trygr.io, google-analytics.com}
IT      docpeter.it         Clearblue test gravidanza pregnancy visual 2 stick                                      {docpeter.it, connectif.cloud, google-analytics.com, magnews.net, google.com}
        efarma.com          Clearblue Test di Gravidanza Con Indicatore delle Settimane 2 Test  {salesmanago.pl, googlesyndication.com, zendesk.com, yotpo.com, google-analytics.com, efarma.com}
        farmae.it           Clearblue Test Di Gravidanza Con Indicatore Delle Settimane 2 Test                                                             {trustpilot.com, google-analytics.com}
        farmasave.it        Clearblue Early Test di Gravidanza Rilevazione Precoce 1 Test                                            {farmasave.it, paypal.com, google-analytics.com, clarity.ms}
        redcare.it          Clearblue Test di Gravidanza con Indicatore delle Settimane 2 pz                                                                            {redcare.it, adtriba.com}
        tuttofarma.it       MyTest Test di Gravidanza a Risultato Rapido 2 Pezzi                                                                     {google-analytics.com, google.com, bing.net}
NL      da.nl               DA Zwangerschapstest 2 stuks                                                                                                                                  {da.nl}
        deonlinedrogist.nl  Clearblue Digitale Zwangerschapstest met Wekenindicator (2st)                                                                                    {deonlinedrogist.nl}

In [40]:
agg_funcs = {
    'req_domain': ['nunique', lambda x: set(x)]
}

result = dflr.groupby(['country', 'site_domain', 'leak_value']).agg(agg_funcs)
result.columns = ['_'.join(col).strip() for col in result.columns.values]
result = result.sort_values(by='req_domain_nunique', ascending=False).head(10)

print(result.to_latex())
# the table need to be cut due to space

\begin{tabular}{lllrl}
\toprule
 &  &  & req_domain_nunique & req_domain_<lambda_0> \\
country & site_domain & leak_value &  &  \\
\midrule
IT & efarma.com & Clearblue Test di Gravidanza Con Indicatore delle Settimane 2 Test & 6 & {'salesmanago.pl', 'googlesyndication.com', 'zendesk.com', 'yotpo.com', 'google-analytics.com', 'efarma.com'} \\
\cline{1-5} \cline{2-5}
\multirow[t]{2}{*}{FR} & docmorris.fr & Gestarelle G3+ Grossesse 90 Capsules & 5 & {'abtasty.com', 'bing.com', 'adsrvr.org', 'bing.net', 'google-analytics.com'} \\
\cline{2-5}
 & soin-et-nature.com & Clearblue Test de Grossesse Digital en vente en pharmacie & 5 & {'doubleclick.net', 'google.fr', 'googleadservices.com', 'soin-et-nature.com', 'google.com'} \\
\cline{1-5} \cline{2-5}
\multirow[t]{2}{*}{IT} & docpeter.it & Clearblue test gravidanza pregnancy visual 2 stick & 5 & {'docpeter.it', 'connectif.cloud', 'google-analytics.com', 'magnews.net', 'google.com'} \\
\cline{2-5}
 & farmasave.it & Clearblue Early Test di Gravida

In [28]:
agg_funcs = {
    'entity': ['nunique', lambda x: set(x)]
}

result = dflr.groupby(['country', 'site_domain', 'leak_value']).agg(agg_funcs)
result.columns = ['_'.join(col).strip() for col in result.columns.values]
result = result.sort_values(by='entity_nunique', ascending=False).head(10)

# print(result.to_latex())
result

entity_nunique  \
country site_domain         leak_value                                                                                              
IT      efarma.com          Clearblue Test di Gravidanza Con Indicatore delle Settimane 2 Test                                  5   
FR      docmorris.fr        Gestarelle G3+ Grossesse 90 Capsules                                                                4   
IT      farmasave.it        Clearblue Early Test di Gravidanza Rilevazione Precoce 1 Test                                       4   
FR      universpharmacie.fr Clearblue Test de Grossesse Détection Rapide X2                                                     3   
ES      farmaciabarata.es   Clearblue Test de Embarazo Ultratemprana, 2 unidades                                                3   
        farmaciasdirect.es  Compra Clearblue Digital Prueba de Embarazo con Indicador de Semanas, 1 Test Digital                3   
IT      1000farmacie.it     Test Gravidanza Babytest Plus 1 1 Pezzo                                                             3   
        docpeter.it         Clearblue test gravidanza pregnancy visual 2 stick                                                  2   
ES      farmaciajimenez.com Femibion 2 Embarazo 28 comprimidos y 28 cápsulas                                                    2   
        boticas23.com       Sanitec Test de Embarazo                                                                            2   

                                                                                                                                               entity_<lambda_0>  
country site_domain         leak_value                                                                                                                            
IT      efarma.com          Clearblue Test di Gravidanza Con Indicatore delle Settimane 2 Test                              {, Yotpo, Zendesk, Benhauer, Google}  
FR      docmorris.fr        Gestarelle G3+ Grossesse 90 Capsules                                                   {AB Tasty, Google, Microsoft, The Trade Desk}  
IT      farmasave.it        Clearblue Early Test di Gravidanza Rilevazione Precoce 1 Test                                          {, PayPal, Google, Microsoft}  
FR      universpharmacie.fr Clearblue Test de Grossesse Détection Rapide X2                                                                {, Google, Microsoft}  
ES      farmaciabarata.es   Clearblue Test de Embarazo Ultratemprana, 2 unidades                                                           {, Google, Microsoft}  
        farmaciasdirect.es  Compra Clearblue Digital Prueba de Embarazo con Indicador de Semanas, 1 Test Digital                          {, Microsoft, Zendesk}  
IT      1000farmacie.it     Test Gravidanza Babytest Plus 1 1 Pezzo                                                 {WebShop Marketing Kft, Rudderstack, Google}  
        docpeter.it         Clearblue test gravidanza pregnancy visual 2 stick                                                                        {, Google}  
ES      farmaciajimenez.com Femibion 2 Embarazo 28 comprimidos y 28 cápsulas                                                                  {Google, Benhauer}  
        boticas23.com       Sanitec Test de Embarazo                                                                                          {Google, Facebook}

In [30]:
dfl.leak_value.value_counts().head(10).to_frame()

,count
leak_value,
Clearblue Test di Gravidanza Rilevazione Rapida 2 Test,92
DA Zwangerschapstest 2 stuks,55
Test Gravidanza Meds 1 Pezzo,52
Clearblue Schwangerschaftstest Ultra Frühtest Dig,47
Clearblue test gravidanza pregnancy visual 2 stick,39
FRÜHTEST Schwangerschaftstest 1 St,38
Alvita Test De Embarazo,37
Clearblue Zwangerschapstest Ultravroeg,36
Gestarelle Grossesse 30 Capsules,34


In [31]:
dfl.leak_type.value_counts().head(10).to_frame()

,count
leak_type,
url_leak,807
post_leak,235


In [32]:
dfl.groupby(['exp_mode', 'consent_mode'])['site_domain'].\
    nunique().to_frame()


site_domain
exp_mode consent_mode             
tracking Accept                 44
         Reject                 38

### Most common hostnames product names were leaked to

In [39]:
dfl.drop_duplicates(subset=['site_domain', 'req_hostname']).req_hostname.value_counts().head(10)

req_hostname
googleads.g.doubleclick.net     24
www.google.com                  23
bat.bing.com                    21
region1.analytics.google.com    20
td.doubleclick.net              17
region1.google-analytics.com    15
www.google-analytics.com         9
analytics.tiktok.com             7
www.googleadservices.com         7
www.google.it                    5
Name: count, dtype: int64

## Which products were leaked most

In [34]:
dfl.drop_duplicates(['site_domain', 'req_hostname', 'leak_value']).leak_value.value_counts().to_frame()

,count
leak_value,
Clearblue Test di Gravidanza Rilevazione Rapida 2 Test,12
Alvita Test De Embarazo,12
Test Gravidanza Meds 1 Pezzo,11
Schwangerschaftstest Schnelltest von apodiscounter 1 stk,10
Clearblue test gravidanza pregnancy visual 2 stick,9
...,...
Clear Blue Pack Digital+ Early 2 Unidades,1
CLEARBLUE TEST DI GRAVIDANZA CON INDICATORE DELLE SETTIMANE 2 TEST,1
dr. original Zwangerschapstest 2 Stuks,1


## All product name leaks

In [35]:
dfl.drop_duplicates(subset=['site_domain', 'exp_mode', 'consent_mode', 'req_hostname', 'leak_type', 'leak_value'])

,site_domain,search_term,leak_type,leak_encoding,leak_value,consent_mode,exp_mode,country,req_hostname,is_tracker,is_cname_tracker,sst,entity,url,post_data,req_domain,third_party
0,apotal.de,CLEARBLUE Schwangerschaftstest frühe Erkennung,url_leak,urlencode,CLEARBLUE Schwangerschaftstest frühe Erkennung,Accept,tracking,DE,bat.bing.com,True,False,False,Microsoft,https://bat.bing.com/action/0?ti=4024282&Ver=2&mid=803c3f28-c58f-4575-bdb8-4c588d08987c&sid=50e7c58088cc11ef8a4b2326ae1182b6...,,bing.com,True
1,apotal.de,CLEARBLUE Schwangerschaftstest frühe Erkennung,url_leak,urlencode,CLEARBLUE Schwangerschaftstest frühe Erkennung,Accept,tracking,DE,www.etracker.de,True,False,False,etracker,https://www.etracker.de/cntcc?&&et=3X9dHm&v=5.0&tc=17287597474736&pagename=CLEARBLUE%20Schwangerschaftstest%20fr%C3%BChe%20E...,,etracker.de,True
4,apotal.de,CLEARBLUE Schwangerschaftstest frühe Erkennung,post_leak,,CLEARBLUE Schwangerschaftstest frühe Erkennung,Accept,tracking,DE,www.etracker.de,True,False,False,etracker,https://www.etracker.de/api/v6/tracking/webEvents,"------WebKitFormBoundaryu94X1ZZVobs5oVfA\r\nContent-Disposition: form-data; name=""et""\r\n\r\n3X9dHm\r\n------WebKitFormBound...",etracker.de,True
5,shop-apotheke.com,RedCare Schwangerschaftstest - Frühtest 1 St,url_leak,base64,RedCare Schwangerschaftstest - Frühtest 1 St,Accept,tracking,DE,o.adtriba.com,True,False,False,,https://o.adtriba.com/collect?id=f38892cb-4d5c-4de9-b4a6-c05871a13e09&ts=1728756131079&data=eyJpZCI6IjNiYzQxOTkzLWJlMmQtNGE1...,,adtriba.com,True
6,shop-apotheke.com,RedCare Schwangerschaftstest - Frühtest 1 St,url_leak,urlencode,RedCare Schwangerschaftstest - Frühtest 1 St,Accept,tracking,DE,klpoz.shop-apotheke.com,True,False,True,Shop-Apotheke,https://klpoz.shop-apotheke.com/g/collect?v=2&tid=G-102RDDVWR0&gtm=45je4a90v9100851417z8810479810za200zb810479810&_p=1728756...,,shop-apotheke.com,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1079,topfarmacia.it,Clearblue Test Gravidanza Con Rilevazione Rapida Confezione Con 1 Test,post_leak,urlencode,Clearblue Test Gravidanza Con Rilevazione Rapida Confezione Con 1 Test,Reject,tracking,IT,region1.google-analytics.com,True,False,False,Google,https://region1.google-analytics.com/g/collect?v=2&tid=G-NP0YYFBBSE&gtm=45je4ae0v9137517540z8897045423za200zb897045423&_p=17...,en=view_item&pr1=id920339076~nmMytest%20Hcg%20Rapid%20Test%20Di%20Gravidanza%202%20Pezzi~lp1~lnOutlet%3A%20Esplora%20i%20pro...,google-analytics.com,True
1081,topfarmacia.it,Clearblue Test Gravidanza Con Rilevazione Rapida Confezione Con 1 Test,url_leak,urlencode,Clearblue Test Gravidanza Con Rilevazione Rapida Confezione Con 1 Test,Reject,tracking,IT,region1.google-analytics.com,True,False,False,Google,https://region1.google-analytics.com/g/collect?v=2&tid=G-J2V1L0GDQH&gtm=45je4ae0v873983442z8897045423za200zb897045423&_p=172...,,google-analytics.com,True
1083,farmacialoreto.it,Test Di Gravidanza - Midstream Large GIMA,url_leak,urlencode,Test Di Gravidanza - Midstream Large GIMA,Reject,tracking,IT,www.google-analytics.com,True,False,False,Google,https://www.google-analytics.com/collect?v=1&_v=j101&aip=1&a=408448368&t=event&ni=1&cu=EUR&_s=4&dl=https%3A%2F%2Ffarmacialor...,,google-analytics.com,True
1084,farmacialoreto.it,Test Di Gravidanza - Midstream Large GIMA,post_leak,urlencode,Test Di Gravidanza - Midstream Large GIMA,Reject,tracking,IT,region1.google-analytics.com,True,False,False,Google,https://region1.google-analytics.com/g/collect?v=2&tid=G-DK66201L06&gtm=45je4a90v874831658za200&_p=1728836439007&gcs=G100&gc...,en=view_promotion&_ee=1&pr1=id973344486~cnlive~cs42108~nmAminoram%20S%20Fresenius%20Kabi%2024%20Bustine~qt1~afFarmacia%20Lor...,google-analytics.com,True


In [ ]:
# count how many websites a particular domain received a leak
dfl.groupby(['site_domain', 'req_hostname']).size().reset_index(name='count').sort_values(by='count', ascending=False)
# df.groupby(['site_domain', 'req_hostname']).size().reset_index(name='count').sort_values(by='count', ascending=False)